In [4]:
# --- JupyterDash + Dash imports ---
from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import dash_leaflet as dl
import plotly.express as px
import pandas as pd
import base64

# --- Your CRUD module ---
# FIX ME if your CRUD file/class name differs
from animal_shelter import AnimalShelter

# ----------------------------
# Database connection settings
# ----------------------------
username = "aacuser"
password = "Pass123j"

db = AnimalShelter(username, password)

# ----------------------------
# Load initial dataset
# ----------------------------
df = pd.DataFrame.from_records(db.read({}))
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)

# ----------------------------
# App setup
# ----------------------------
app = JupyterDash(__name__)

# ----------------------------
# Logo (optional)
# ----------------------------
# Put your image file in the same folder as the notebook
image_filename = "Grazioso Salvare Logo.png"
encoded_image = None
try:
    encoded_image = base64.b64encode(open(image_filename, "rb").read()).decode()
except FileNotFoundError:
    encoded_image = None

# ----------------------------
# Layout
# ----------------------------
app.layout = html.Div([
    html.Center(html.H1("CS-340 Dashboard")),
    html.Hr(),

    # Header row: logo + identifier + filter radio buttons
    html.Div([
        html.Div([
            html.Img(
                src=f"data:image/png;base64,{encoded_image}" if encoded_image else "",
                style={"height": "90px", "display": "block"} if encoded_image else {"display": "none"}
            ),
            html.H3("Chris P - Grazioso Salvare Dashboard"),
        ], style={"width": "35%", "display": "inline-block", "verticalAlign": "top"}),

        html.Div([
            html.H4("Rescue Type Filter"),
            dcc.RadioItems(
                id="filter-type",
                options=[
                    {"label": "Reset (All)", "value": "reset"},
                    {"label": "Water Rescue", "value": "water"},
                    {"label": "Mountain/Wilderness Rescue", "value": "mountain"},
                    {"label": "Disaster/Individual Tracking", "value": "disaster"},
                ],
                value="reset",
                inline=True
            )
        ], style={"width": "60%", "display": "inline-block", "paddingLeft": "20px"})
    ]),

    html.Hr(),

    # Data table
    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[],
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "minWidth": "120px", "maxWidth": "250px", "whiteSpace": "normal"},
    ),

    html.Br(),

    # Chart + Map side by side
    html.Div([
        html.Div(id="graph-id", style={"width": "48%", "display": "inline-block", "verticalAlign": "top"}),
        html.Div(id="map-id", style={"width": "48%", "display": "inline-block", "paddingLeft": "2%", "verticalAlign": "top"}),
    ]),
])

# ---------------------------------------
# Callback 1: Radio filter -> Mongo query
# ---------------------------------------
@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value")
)
def update_dashboard(filter_type):
    query = {}

    # IMPORTANT: Replace these breeds with the ones required by your rubric/spec
    if filter_type == "water":
        query = {"breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]}}

    elif filter_type == "mountain":
        query = {"breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Bernese Mountain Dog"]}}

    elif filter_type == "disaster":
        query = {"breed": {"$in": ["Bloodhound", "Belgian Malinois", "Doberman Pinscher"]}}

    # reset => query stays {}

    dff = pd.DataFrame.from_records(db.read(query))
    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    return dff.to_dict("records")

# ---------------------------------------
# Callback 2: Chart updates from table view
# ---------------------------------------
@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_graphs(viewData):
    if not viewData:
        return html.Div("No data to display.")

    dff = pd.DataFrame.from_dict(viewData)

    # Robust: only run if breed exists
    if "breed" not in dff.columns:
        return html.Div("No 'breed' column found for chart.")

    # Top 10 breeds bar chart (cleaner than a huge histogram)
    counts = dff["breed"].value_counts().nlargest(10).reset_index()
    counts.columns = ["breed", "count"]

    fig = px.bar(counts, x="breed", y="count", title="Top 10 Breeds (Current Table View)")
    fig.update_layout(xaxis_title="Breed", yaxis_title="Count")

    return dcc.Graph(figure=fig)

# ---------------------------------------
# Callback 3: Map updates from selected row
# ---------------------------------------
@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(viewData, selected_rows):
    if not viewData:
        return html.Div("No data to map.")

    dff = pd.DataFrame.from_dict(viewData)

    # Ensure lat/long columns exist
    if "location_lat" not in dff.columns or "location_long" not in dff.columns:
        return html.Div("No location_lat/location_long columns found.")

    # Drop missing coords
    dff = dff.dropna(subset=["location_lat", "location_long"])
    if dff.empty:
        return html.Div("No valid coordinates in current table view.")

    # Default selection to first visible row
    row = 0 if not selected_rows else selected_rows[0]
    if row >= len(dff):
        row = 0

    lat = float(dff.iloc[row]["location_lat"])
    lon = float(dff.iloc[row]["location_long"])

    breed = str(dff.iloc[row]["breed"]) if "breed" in dff.columns else "Animal"
    name = str(dff.iloc[row]["name"]) if "name" in dff.columns else ""

    return dl.Map(
        style={"width": "100%", "height": "500px"},
        center=[lat, lon],   # center on selected animal (important!)
        zoom=11,
        children=[
            dl.TileLayer(),
            dl.Marker(
                position=[lat, lon],
                children=[
                    dl.Tooltip(breed),
                    dl.Popup([
                        html.B("Animal: "), html.Span(name),
                        html.Br(),
                        html.B("Breed: "), html.Span(breed),
                        html.Br(),
                        html.B("Location: "), html.Span(f"{lat}, {lon}")
                    ])
                ]
            )
        ]
    )

# ----------------------------
# Run (best for Jupyter)
# ----------------------------
app.run_server(mode="external", debug=True)

Dash app running on https://shelfinvent-lucascircle-3000.codio.io/proxy/8050/
